## TECH CHALLENGE - FASE 3: Exploração do Dataset MedQuAD

Este notebook tem como objetivo explorar a estrutura do dataset MedQuAD, entender o formato dos arquivos XML e preparar os dados para uso no
assistente médico que foi requisitado para fase 3.

Nesta etapa, serão realizadas as seguintes atividades:

- localização dos arquivos do dataset no projeto
- inspeção inicial da estrutura dos arquivos XML
- identificação dos campos de pergunta e resposta
- extração estruturada dos dados
- preparação de um arquivo processado para uso posterior em RAG e fine-tuning

O foco inicial será a coleção `1_CancerGov_QA`, por estar mais alinhada
ao tema já trabalhado nas fases anteriores do projeto.

In [2]:
from pathlib import Path
import xml.etree.ElementTree as ET
import json
import pandas as pd

In [3]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.rag.documents_loader import load_medquad_documents
from src.rag.vector_store import build_vector_store, save_vector_store
from src.rag.retriever import get_retriever
from src.llm.ollama_client import get_llm
from src.assistant.medical_assistant import answer_medical_question
from src.security.guardrails import evaluate_question_risk, build_guardrail_response


ImportError: cannot import name 'is_safe_question' from 'src.security.guardrails' (C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\src\security\guardrails.py)

## 1. Definição do caminho do dataset

Nesta etapa, será definido o caminho da pasta onde foi armazenado o
dataset MedQuAD dentro do projeto.

Inicialmente, a exploração será feita na coleção `1_CancerGov_QA`,
pois ela contém perguntas e respostas relacionadas ao domínio oncológico,
o que torna essa base mais aderente ao contexto do projeto.

In [3]:
base_path = Path("../data/medical_qa/raw/MedQuAD/1_CancerGov_QA")
files = list(base_path.glob("*.xml"))

print(f"Total de arquivos XML encontrados: {len(files)}")
print("\nPrimeiros 8 arquivos:")
for f in files[:8]:
    print(f.name)

Total de arquivos XML encontrados: 116

Primeiros 8 arquivos:
0000001_1.xml
0000001_2.xml
0000001_3.xml
0000001_4.xml
0000001_5.xml
0000001_6.xml
0000001_7.xml
0000003_1.xml


## 2. Inspeção inicial de um arquivo XML

Um arquivo será selecionado para inspeção manual.
Essa etapa é importante para compreender a estrutura bruta do XML e verificar como as informações estão organizadas internamente.

In [4]:
sample_file = files[0]
sample_file

WindowsPath('../data/medical_qa/raw/MedQuAD/1_CancerGov_QA/0000001_1.xml')

In [5]:
with open(sample_file, "r", encoding="utf-8") as f: content = f.read()

print(content[:2000])

<?xml version="1.0" encoding="UTF-8"?>
<Document id="0000001_1" source="CancerGov" url="https://www.cancer.gov/types/leukemia/patient/adult-all-treatment-pdq">
<Focus>Adult Acute Lymphoblastic Leukemia</Focus>
<FocusAnnotations>
	<UMLS>
		<CUIs>
			<CUI>C0751606</CUI>
		</CUIs>
		<SemanticTypes>
			<SemanticType>T191</SemanticType>
		</SemanticTypes>
		<SemanticGroup>Disorders</SemanticGroup>
	</UMLS>
</FocusAnnotations>
<QAPairs>
	<QAPair pid="1">
			<Question qid="0000001_1-1" qtype="information">What is (are) Adult Acute Lymphoblastic Leukemia ?</Question>
			<Answer>Key Points
                    - Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of white blood cell).    - Leukemia may affect red blood cells, white blood cells, and platelets.    - Previous chemotherapy and exposure to radiation may increase the risk of developing ALL.    - Signs and symptoms of adult ALL include fever, feeling tired, and easy b

## 3. Leitura da estrutura XML

Após observar o conteúdo bruto do arquivo, será feito o parsing do XML
com a biblioteca `xml.etree.ElementTree`.

O objetivo desta etapa é identificar a tag raiz e compreender a organização
hierárquica dos elementos presentes no arquivo.

In [6]:
tree = ET.parse(sample_file)
root = tree.getroot()

print("Tag raiz:", root.tag)
print("\nFilhos imediatos da raiz:")
for child in root:
    print(child.tag)

Tag raiz: Document

Filhos imediatos da raiz:
Focus
FocusAnnotations
QAPairs


## 4. Identificação das tags existentes

Nesta etapa, serão listadas as tags encontradas no arquivo XML para
identificar onde estão armazenadas as perguntas e as respostas.

Esse passo é importante porque permite construir um parser adequado
para a estrutura específica do MedQuAD.

In [7]:
tags = set()

for elem in root.iter():
    tags.add(elem.tag)

print("Tags encontradas no XML:")
for tag in sorted(tags):
    print(tag)

Tags encontradas no XML:
Answer
CUI
CUIs
Document
Focus
FocusAnnotations
QAPair
QAPairs
Question
SemanticGroup
SemanticType
SemanticTypes
UMLS


## 5. Inspeção dos elementos textuais

Agora serão exibidos os elementos do XML que possuem conteúdo textual.
Essa inspeção ajuda a localizar os campos que efetivamente armazenam
as perguntas e respostas do dataset.

In [8]:
for elem in root.iter():
    if elem.text and elem.text.strip():
        print(f"{elem.tag} => {elem.text.strip()[:300]}")
        print("-" * 80)

Focus => Adult Acute Lymphoblastic Leukemia
--------------------------------------------------------------------------------
CUI => C0751606
--------------------------------------------------------------------------------
SemanticType => T191
--------------------------------------------------------------------------------
SemanticGroup => Disorders
--------------------------------------------------------------------------------
Question => What is (are) Adult Acute Lymphoblastic Leukemia ?
--------------------------------------------------------------------------------
Answer => Key Points
                    - Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of white blood cell).    - Leukemia may affect red blood cells, white blood cells, and platelets.    - Previous chemotherapy and exposure to radia
--------------------------------------------------------------------------------
Question => What are the symptom

## 6. Criação da função de extração de pares pergunta-resposta

Com base na estrutura observada no XML, será criada uma função para extrair os pares de pergunta e resposta de cada arquivo.

Essa função permitirá automatizar a leitura de múltiplos arquivos da coleção selecionada.

In [9]:
def parse_medquad_xml(sample_file):
    tree = ET.parse(sample_file)
    root = tree.getroot()

    records = []
    file_name = Path(sample_file).name

    for idx, qa_pair in enumerate(root.findall(".//QAPair")):
        question = qa_pair.findtext("Question")
        answer = qa_pair.findtext("Answer")

        if question and answer:
            records.append({
                "id": f"{file_name}_{idx}",
                "source_file": file_name,
                "collection": "CancerGov",
                "question": question.strip(),
                "answer": answer.strip()
            })

    return records

## 7. Teste da função em um arquivo de exemplo

Nesta etapa, a função criada será aplicada a um único arquivo XML, com o objetivo de validar se a extração dos pares pergunta-resposta está ocorrendo corretamente.

In [10]:
records = parse_medquad_xml(sample_file)

print(f"Total de pares extraídos do arquivo de exemplo: {len(records)}")
records[:2]

Total de pares extraídos do arquivo de exemplo: 7


[{'id': '0000001_1.xml_0',
  'source_file': '0000001_1.xml',
  'collection': 'CancerGov',
  'question': 'What is (are) Adult Acute Lymphoblastic Leukemia ?',
  'answer': 'Key Points\n                    - Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of white blood cell).    - Leukemia may affect red blood cells, white blood cells, and platelets.    - Previous chemotherapy and exposure to radiation may increase the risk of developing ALL.    - Signs and symptoms of adult ALL include fever, feeling tired, and easy bruising or bleeding.     - Tests that examine the blood and bone marrow are used to detect (find) and diagnose adult ALL.    - Certain factors affect prognosis (chance of recovery) and treatment options.\n                \n                \n                    Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of white blood cell).\n

## 8. Extração dos dados de todos os arquivos da coleção

Após validar a função em um arquivo de exemplo, será feita a extração dos dados de todos os arquivos XML da coleção `1_CancerGov_QA`.

Ao final, será gerada uma lista consolidada com todos os pares pergunta-resposta encontrados.

In [11]:
all_records = []

for file_path in files:
    try:
        extracted = parse_medquad_xml(file_path)
        all_records.extend(extracted)
    except Exception as e:
        print(f"Erro ao processar {file_path.name}: {e}")

print(f"Total de registros extraídos: {len(all_records)}")

Total de registros extraídos: 729


## 9. Conversão para DataFrame

Nesta etapa, os registros extraídos serão convertidos para um DataFrame,
facilitando a visualização, inspeção e futuras etapas de pré-processamento.

In [12]:
df_medquad = pd.DataFrame(all_records)

print(df_medquad.shape)
df_medquad.head()

(729, 5)


,id,source_file,collection,question,answer
0,0000001_1.xml_0,0000001_1.xml,CancerGov,What is (are) Adult Acute Lymphoblastic Leukem...,Key Points\n - Adult acute ...
1,0000001_1.xml_1,0000001_1.xml,CancerGov,What are the symptoms of Adult Acute Lymphobla...,"Signs and symptoms of adult ALL include fever,..."
2,0000001_1.xml_2,0000001_1.xml,CancerGov,How to diagnose Adult Acute Lymphoblastic Leuk...,Tests that examine the blood and bone marrow a...
3,0000001_1.xml_3,0000001_1.xml,CancerGov,What is the outlook for Adult Acute Lymphoblas...,Certain factors affect prognosis (chance of re...
4,0000001_1.xml_4,0000001_1.xml,CancerGov,Who is at risk for Adult Acute Lymphoblastic L...,Previous chemotherapy and exposure to radiatio...


## 10. Verificação de valores ausentes e duplicados

Antes de salvar o dataset processado, é importante realizar uma verificação
simples de qualidade dos dados, observando a presença de valores nulos
e registros duplicados.

In [13]:
print("Valores nulos:")
print(df_medquad.isnull().sum())

print("\nRegistros duplicados:")
print(df_medquad.duplicated().sum())

Valores nulos:
id             0
source_file    0
collection     0
question       0
answer         0
dtype: int64

Registros duplicados:
0


## 11. Salvamento do dataset processado

Por fim, o conjunto de dados tratado será salvo na pasta `processed`,
permitindo seu reaproveitamento nas próximas fases do pipeline,
como construção da base vetorial, RAG e fine-tuning.

In [14]:
json_path = "../data/medical_qa/processed/medquad_cancergov_qa.json"
documents = load_medquad_documents(json_path)

print("Total de documentos:", len(documents))
documents[0]

Total de documentos: 812


Document(metadata={'id': '0000001_1.xml_0', 'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'source': 'MedQuAD'}, page_content='Question: What is (are) Adult Acute Lymphoblastic Leukemia ?\nAnswer: Key Points\n                    - Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of white blood cell).    - Leukemia may affect red blood cells, white blood cells, and platelets.    - Previous chemotherapy and exposure to radiation may increase the risk of developing ALL.    - Signs and symptoms of adult ALL include fever, feeling tired, and easy bruising or bleeding.     - Tests that examine the blood and bone marrow are used to detect (find) and diagnose adult ALL.    - Certain factors affect prognosis (chance of recovery) and treatment options.\n                \n                \n                    Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymph

## 12. Visualização de exemplos finais

Para concluir esta etapa, serão exibidos alguns exemplos do dataset processado, permitindo validar visualmente o resultado da extração.

In [15]:
for i, row in df_medquad.head(3).iterrows():
    print(f"Registro {i+1}")
    print(f"Arquivo de origem: {row['source_file']}")
    print(f"Pergunta: {row['question']}")
    print(f"Resposta: {row['answer'][:500]}")
    print("=" * 100)

Registro 1
Arquivo de origem: 0000001_1.xml
Pergunta: What is (are) Adult Acute Lymphoblastic Leukemia ?
Resposta: Key Points
                    - Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of white blood cell).    - Leukemia may affect red blood cells, white blood cells, and platelets.    - Previous chemotherapy and exposure to radiation may increase the risk of developing ALL.    - Signs and symptoms of adult ALL include fever, feeling tired, and easy bruising or bleeding.     - Tests that examine the blood and bone marrow are u
Registro 2
Arquivo de origem: 0000001_1.xml
Pergunta: What are the symptoms of Adult Acute Lymphoblastic Leukemia ?
Resposta: Signs and symptoms of adult ALL include fever, feeling tired, and easy bruising or bleeding. The early signs and symptoms of ALL may be like the flu or other common diseases. Check with your doctor if you have any of the following:         - Weakness or feel

## 13. Preparação de documentos para RAG

Nesta etapa os pares pergunta-resposta serão convertidos em documentos
que poderão ser indexados em uma base vetorial.

Cada documento conterá:
- pergunta
- resposta
- metadados da fonte

In [16]:
from langchain_core.documents import Document

documents = []

for _, row in df_medquad.iterrows():

    text = f"""
Question: {row['question']}
Answer: {row['answer']}
"""

    metadata = {
        "source_file": row["source_file"],
        "collection": row["collection"],
        "id": row["id"]
    }

    documents.append(
        Document(
            page_content=text,
            metadata=metadata
        )
    )

print("Total de documentos criados:", len(documents))
documents[0]

Total de documentos criados: 729


Document(metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_0'}, page_content='\nQuestion: What is (are) Adult Acute Lymphoblastic Leukemia ?\nAnswer: Key Points\n                    - Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of white blood cell).    - Leukemia may affect red blood cells, white blood cells, and platelets.    - Previous chemotherapy and exposure to radiation may increase the risk of developing ALL.    - Signs and symptoms of adult ALL include fever, feeling tired, and easy bruising or bleeding.     - Tests that examine the blood and bone marrow are used to detect (find) and diagnose adult ALL.    - Certain factors affect prognosis (chance of recovery) and treatment options.\n                \n                \n                    Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of w

In [17]:
print(f"Total de documents: {len(documents)}\n")

for i, doc in enumerate(documents[:10], 1):
    print(f"--- Documento {i} ---")
    print("ID:", doc.metadata.get("id"))
    print("Arquivo:", doc.metadata.get("source_file"))
    print("Coleção:", doc.metadata.get("collection"))
    print("Conteúdo:")
    print(doc.page_content[:500])
    print("\n")

Total de documents: 729

--- Documento 1 ---
ID: 0000001_1.xml_0
Arquivo: 0000001_1.xml
Coleção: CancerGov
Conteúdo:

Question: What is (are) Adult Acute Lymphoblastic Leukemia ?
Answer: Key Points
                    - Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of white blood cell).    - Leukemia may affect red blood cells, white blood cells, and platelets.    - Previous chemotherapy and exposure to radiation may increase the risk of developing ALL.    - Signs and symptoms of adult ALL include fever, feeling tired, and easy bruising


--- Documento 2 ---
ID: 0000001_1.xml_1
Arquivo: 0000001_1.xml
Coleção: CancerGov
Conteúdo:

Question: What are the symptoms of Adult Acute Lymphoblastic Leukemia ?
Answer: Signs and symptoms of adult ALL include fever, feeling tired, and easy bruising or bleeding. The early signs and symptoms of ALL may be like the flu or other common diseases. Check with your doctor if you ha

In [18]:
source_files = [doc.metadata.get("source_file") for doc in documents]
unique_files = set(source_files)

print("Total de documentos:", len(documents))
print("Total de arquivos únicos:", len(unique_files))
print("\nAlguns arquivos únicos:")
print(list(unique_files)[:20])

Total de documentos: 729
Total de arquivos únicos: 116

Alguns arquivos únicos:
['0000020_1.xml', '0000028_1.xml', '0000004_6.xml', '0000019_5.xml', '0000026_2.xml', '0000003_4.xml', '0000031_2.xml', '0000040_1.xml', '0000024_6.xml', '0000006_6.xml', '0000013_3_4.xml', '0000027_4.xml', '0000001_7.xml', '0000007_4.xml', '0000004_4.xml', '0000024_9.xml', '0000032_2.xml', '0000043_1.xml', '0000004_3.xml', '0000004_5.xml']


In [19]:
matches = []

for doc in documents:
    text = doc.page_content.lower()
    if "breast cancer" in text:
        matches.append(doc)

print(f"Total com 'breast cancer': {len(matches)}\n")

for i, doc in enumerate(matches[:5], 1):
    print(f"--- Match {i} ---")
    print("ID:", doc.metadata.get("id"))
    print("Arquivo:", doc.metadata.get("source_file"))
    print(doc.page_content[:500])
    print("\n")

Total com 'breast cancer': 45

--- Match 1 ---
ID: 0000004_6.xml_7
Arquivo: 0000004_6.xml

Question: What are the treatments for Childhood Hodgkin Lymphoma ?
Answer: Key Points
                    - There are different types of treatment for children with Hodgkin lymphoma.     - Children with Hodgkin lymphoma should have their treatment planned by a team of health care providers who are experts in treating childhood cancer.    - Children and adolescents may have treatment-related side effects that appear months or years after treatment for Hodgkin lymphoma.     - Five types of standa


--- Match 2 ---
ID: 0000006_1.xml_0
Arquivo: 0000006_1.xml

Question: What is (are) Adult Central Nervous System Tumors ?
Answer: Key Points
                    - An adult central nervous system tumor is a disease in which abnormal cells form in the tissues of the brain and/or spinal cord.    - A tumor that starts in another part of the body and spreads to the brain is called a metastatic brain tumor.   

## 14. Criação de Chunks para processamento da base vetorial

In [20]:
import sys
!{sys.executable} -m pip install -U langchain-text-splitters


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 15. Criação da base vetorial

Para permitir recuperação semântica das informações médicas,
os documentos serão convertidos em embeddings e armazenados
em uma base vetorial utilizando FAISS.

In [21]:
vector_store, num_chunks = build_vector_store(documents)

print(f"Chunks criados: {num_chunks}")

save_vector_store(vector_store, "vector_store")

Chunks criados: 8811
Base vetorial salva em: vector_store


O uso de chunking foi necessário para evitar que textos maiores excedam o limite de entrada do modelo de embeddings.
A estratégia escolhida foi o RecursiveCharacterTextSplitter, recomendado pelo LangChain para textos genéricos, com sobreposição entre chunks para
preservar contexto semântico.

## 16. Teste de recuperação semântica

Após a criação da base vetorial, será testado o mecanismo de recuperação
para verificar se a busca semântica retorna trechos relevantes do dataset.

In [22]:
retriever = get_retriever(vector_store, k=3)

In [23]:
query = "What are the symptoms of breast cancer?"

docs = retriever.invoke(query)

print(f"Total recuperado: {len(docs)}\n")

for i, doc in enumerate(docs, 1):
    print(f"--- Documento {i} ---")
    print("ID:", doc.metadata.get("id"))
    print("Arquivo:", doc.metadata.get("source_file"))
    print("Coleção:", doc.metadata.get("collection"))
    print("Trecho:")
    print(doc.page_content[:800])
    print("\n")

Total recuperado: 3

--- Documento 1 ---
ID: 0000027_3.xml_2
Arquivo: 0000027_3.xml
Coleção: CancerGov
Trecho:
Question: What are the symptoms of Breast Cancer ?
Answer: Signs of breast cancer include a lump or change in the breast.


--- Documento 2 ---
ID: 0000027_2.xml_3
Arquivo: 0000027_2.xml
Coleção: CancerGov
Trecho:
Question: What are the symptoms of Male Breast Cancer ?
Answer: Men with breast cancer usually have lumps that can be felt.Lumps and other signs may be caused by male breast cancer or by other conditions. Check with your doctor if you notice a change in your breasts.


--- Documento 3 ---
ID: 0000027_1.xml_4
Arquivo: 0000027_1.xml
Coleção: CancerGov
Trecho:
Answer: Signs of breast cancer include a lump or change in the breast. These and other signs may be caused by breast cancer or by other conditions. Check with your doctor if you have any of the following:         - A lump or thickening in or near the breast or in the underarm area.    - A change in the size or sha

## 17. Teste de geração com LLM local (Ollama)

Nesta etapa, os documentos recuperados pelo retriever serão utilizados como contexto para um modelo de linguagem local executado via Ollama.

O objetivo é gerar respostas médicas em linguagem natural utilizando apenas o conteúdo recuperado da base de conhecimento.

In [24]:
llm = get_llm(model_name="mistral", temperature=0.2)

C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\src\llm\ollama_client.py:4: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  return ChatOllama(


In [25]:
question = "What are the symptoms of breast cancer?"

result = answer_medical_question(
    question=question,
    llm=llm,
    retriever=retriever
)

print(result["answer"])
print("\nFontes:")
for source in result["sources"]:
    print(source)

 Based on the provided context, symptoms of both female and male breast cancer may include a lump or change in the breast. Other signs might be a change in the size or shape of the breast, a dimple or puckering in the skin of the breast, a nipple turned inward into the breast, and fluid, other than breast milk, from the nipple [0000027_1.xml]. It's essential to consult with a healthcare professional if any of these symptoms are noticed, as they could be caused by various conditions.

Fonte: 0000027_1.xml

Fontes:
{'label': 'Fonte 1', 'source': 'desconhecida', 'collection': 'CancerGov', 'source_file': '0000027_3.xml', 'id': '0000027_3.xml_2'}
{'label': 'Fonte 2', 'source': 'desconhecida', 'collection': 'CancerGov', 'source_file': '0000027_2.xml', 'id': '0000027_2.xml_3'}
{'label': 'Fonte 3', 'source': 'desconhecida', 'collection': 'CancerGov', 'source_file': '0000027_1.xml', 'id': '0000027_1.xml_4'}


A resposta gerada nesta etapa já representa um fluxo básico de RAG
(Retrieval-Augmented Generation), no qual o modelo de linguagem utiliza
os documentos recuperados da base vetorial como contexto para formular
a resposta.

## 18. Explainability: indicação das fontes consultadas

Para aumentar a rastreabilidade da resposta gerada, o sistema apresentará as fontes dos documentos recuperados pelo mecanismo de RAG.
Essa estratégia contribui para explainability, permitindo identificar quais registros do dataset foram utilizados como base para a resposta.

In [28]:
def format_sources(docs):
    sources = []

    for i, doc in enumerate(docs):
        metadata = doc.metadata if hasattr(doc, "metadata") else doc.get("metadata", {})

        sources.append({
            "label": f"Fonte {i+1}",
            "collection": metadata.get("collection", "desconhecida"),
            "source_file": metadata.get("source_file", "desconhecido"),
            "id": metadata.get("id", f"doc_{i+1}")
        })

    return sources

In [29]:
def build_explainable_answer(question, retriever, llm):
    
    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are a medical assistant.

Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}

Answer:
"""

    response = llm.invoke(prompt)

    answer = response.content if hasattr(response, "content") else str(response)

    sources = format_sources(docs)

    return {
        "answer": answer,
        "sources": sources
    }

Nesta implementação, a LLM é responsável apenas pela geração textual da resposta. Já a listagem das fontes é montada programaticamente em Python, evitando que o modelo invente referências inexistentes.
Essa separação melhora a confiabilidade da saída final.

## 19. Criação de uma função de consulta ao assistente médico

Para facilitar testes e futuras integrações com a estrutura modular do projeto,
será criada uma função que executa o pipeline completo:

1. recebe a pergunta do usuário
2. recupera documentos relevantes
3. monta o contexto
4. gera a resposta com a LLM
5. exibe as fontes consultadas

In [30]:
def ask_medical_assistant(question, retriever, llm, k=3):
    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are a medical educational assistant focused on breast cancer information.

Use only the context below to answer the question.
Do not invent information.
If the context is insufficient, clearly say that the available sources are insufficient.
Do not provide a definitive diagnosis.
Do not prescribe treatment.
Always answer in a clear and educational tone.

Context:
{context}

Question:
{question}
"""

    response = llm.invoke(prompt)

    final_response = f"""
{response.content}

Sources consulted:
{format_sources(docs)}
""".strip()

    return final_response

In [31]:
question_test = "What are the symptoms of breast cancer?"
print(ask_medical_assistant(question_test, retriever, llm))

Signs of breast cancer include a lump or change in the breast. These and other signs may be caused by breast cancer or by other conditions. Some specific symptoms to watch for include:
- A lump or thickening in or near the breast or in the underarm area.
- A change in the size or shape of the breast.
- A dimple or puckering in the skin of the breast.
- A nipple turned inward into the breast.
- Fluid, other than breast milk, from the nipple, especially if it's bloody.

It is important to note that these symptoms may not always be present and can also be caused by benign conditions. If you notice any changes in your breasts, it is recommended to consult with a healthcare professional for further evaluation.

Sources consulted:
[{'label': 'Fonte 1', 'collection': 'CancerGov', 'source_file': '0000027_3.xml', 'id': '0000027_3.xml_2'}, {'label': 'Fonte 2', 'collection': 'CancerGov', 'source_file': '0000027_2.xml', 'id': '0000027_2.xml_3'}, {'label': 'Fonte 3', 'collection': 'CancerGov', 'sou

## 20. Avaliação qualitativa inicial do pipeline RAG

Nesta etapa, o sistema já é capaz de:

- recuperar trechos semanticamente relevantes do MedQuAD
- utilizar esses trechos como contexto para a geração da resposta
- apresentar as fontes consultadas
- operar localmente via Ollama, sem dependência de APIs pagas

Esse resultado representa a primeira versão funcional do assistente médico
virtual proposto para a Fase 3.

In [4]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from src.security.guardrails import evaluate_question_risk, build_guardrail_response

questions = [
    "What are the symptoms of breast cancer?",
    "Does this mean cancer?",
    "What medicine should I take for breast cancer?"
]

for q in questions:
    result = evaluate_question_risk(q)
    print("Pergunta:", q)
    print("Resultado:", result)
    print("Resposta de segurança:", build_guardrail_response(result))
    print("-" * 80)

Pergunta: What are the symptoms of breast cancer?
Resultado: {'safe': True, 'risk_level': 'low', 'action': 'allow', 'reason': 'Pergunta informativa.'}
Resposta de segurança: 
--------------------------------------------------------------------------------
Pergunta: Does this mean cancer?
Resultado: {'safe': True, 'risk_level': 'medium', 'action': 'allow_with_warning', 'reason': 'Pergunta com interpretação clínica sensível.'}
Resposta de segurança: This question involves sensitive medical interpretation. I can provide general informational support, but this does not replace professional medical evaluation.
--------------------------------------------------------------------------------
Pergunta: What medicine should I take for breast cancer?
Resultado: {'safe': False, 'risk_level': 'high', 'action': 'block', 'reason': 'Pergunta com solicitação de tratamento, medicação ou dosagem.'}
Resposta de segurança: I cannot provide medical treatment, prescription, or dosage guidance. Please consul

In [6]:

def ask_medical_assistant(question, retriever, llm, k=3):
    risk = evaluate_question_risk(question)

    if risk["action"] == "block":
        return build_guardrail_response(risk)

    docs = retriever.invoke(question)

    context = "\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are a medical educational assistant focused on breast cancer information.

Use only the context below to answer the question.
Do not invent information.
If the context is insufficient, clearly say that the available sources are insufficient.
Do not provide a definitive diagnosis.
Do not prescribe treatment.
Do not prescribe dosage.
Always answer in a clear and educational tone.

Context:
{context}

Question:
{question}
"""

    response = llm.invoke(prompt)
    answer = response.content if hasattr(response, "content") else str(response)

    warning = ""
    if risk["action"] == "allow_with_warning":
        warning = build_guardrail_response(risk) + "\n\n"

    final_response = f"""
{warning}{answer}

Sources consulted:
{format_sources(docs)}
""".strip()

    return final_response

In [7]:
print(ask_medical_assistant("What are the symptoms of breast cancer?", retriever, llm))
print(ask_medical_assistant("Does this mean cancer?", retriever, llm))
print(ask_medical_assistant("What medicine should I take for breast cancer?", retriever, llm))

NameError: name 'retriever' is not defined

In [ ]:
question = "What medicine should I take for breast cancer?"

result = answer_medical_question(
    question=question,
    llm=llm,
    retriever=retriever
)

print(result["answer"])
print(result["status"])
print(result["risk_level"])